# Notebook 04: Training — SUM Condition (Baseline)

**Agency Calculus Empirical Validation — Paper C**

Social planner objective: **R = Σu_i (utilitarian sum)**

This is the AI Economist default. Permits unbounded compensation:
gains for high-utility agents can offset losses for low-utility agents.

**Predicted outcomes:**
- Floor utility: declines over training (sacrificed for aggregate)
- Gini coefficient: increases (inequality grows)
- Tax on floor agent: extractive (planner taxes floor to maximize sum)
- Total utility: highest of three conditions

5 seeds × 10M steps each. Expected runtime: ~2-4 hours per seed on Colab GPU.\n\nImportant: this notebook assumes your AI Economist fork already applies the `sum` planner objective inside the real env/scenario reward path. The local `training.py` callback only logs diagnostics.

In [1]:
# ── Environment check ─────────────────────────────────────────────────────
# If ai_economist is missing, run notebook 01 first (it handles installation
# and the required kernel restart).
import sys, os

try:
    import ai_economist  # noqa: F401
except ModuleNotFoundError:
    raise SystemExit(
        "\n❌  ai_economist not found. Run notebook 01_setup_and_test first,\n"
        "    restart the kernel, then return here."
    )

# Add src/ to path
for candidate in [
    '/content/ac-validation/src',
    os.path.join(os.getcwd(), '..', 'src'),
    os.path.join(os.getcwd(), 'src'),
]:
    if os.path.exists(candidate) and candidate not in sys.path:
        sys.path.insert(0, candidate)
        print(f'src on path: {candidate}')
        break


Inside covid19_components.py: 0 GPUs are available.
No GPUs found! Running the simulation on a CPU.
Inside covid19_env.py: 0 GPUs are available.
No GPUs found! Running the simulation on a CPU.
src on path: C:\Users\crens\Documents\GitHub\ac-validation\notebooks\..\src


In [2]:
import sys, os
import numpy as np

sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

from training import run_condition, run_training, TOTAL_TIMESTEPS, N_SEEDS
from metrics import MetricsLogger
import matplotlib.pyplot as plt

CONDITION = 'sum'
RESULTS_DIR = '../results'
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Condition: {CONDITION}')
print(f'Total timesteps per seed: {TOTAL_TIMESTEPS:,}')
print(f'Number of seeds: {N_SEEDS}')
print('Fork requirement: planner reward must be swapped in AI Economist, not only logged in training.py')

Condition: sum
Total timesteps per seed: 10,000,000
Number of seeds: 5
Fork requirement: planner reward must be swapped in AI Economist, not only logged in training.py


## Option A: Full Training (5 seeds × 10M steps)

Recommended for Colab GPU sessions. Run each seed in a separate runtime after a short smoke test.

In [ ]:
# Run a single seed (set SEED = 0, 1, 2, 3, 4 across separate sessions)
SEED = 0  # Change this for each session

# Check if already completed
save_path = f'{RESULTS_DIR}/{CONDITION}_seed{SEED}_metrics.npz'
if os.path.exists(save_path):
    print(f'Seed {SEED} already complete: {save_path}')
    print('Load it with: MetricsLogger.load(save_path)')
else:
    print(f'Starting training: condition={CONDITION}, seed={SEED}')
    logger = run_training(
        condition=CONDITION,
        seed=SEED,
        total_timesteps=TOTAL_TIMESTEPS,
        results_dir=RESULTS_DIR,
    )
    print(f'Training complete. Saved to {save_path}')

Starting training: condition=sum, seed=0

Training: condition=sum, seed=0
Total steps: 10,000,000  Eval every: 100,000


2026-03-18 17:28:02,322	INFO worker.py:1553 -- Started a local Ray instance.


Inside env_wrapper.py: 0 GPUs are available.
No GPUs found! Running the simulation on a CPU.


C:\Users\crens\Documents\GitHub\ac-validation\.venv310\lib\site-packages\numpy\core\numeric.py:330: RuntimeWarning: invalid value encountered in cast
  multiarray.copyto(a, fill_value, casting='unsafe')
2026-03-18 17:28:05,815	INFO algorithm.py:506 -- Current log_level is WARN. For more information, set 'log_level': 'INFO' / 'DEBUG' or use the -v and -vv flags.
2026-03-18 17:28:12,008	ERROR actor_manager.py:496 -- Ray error, taking actor 1 out of service. The actor died because of an error raised in its creation task, ray::RolloutWorker.__init__() (pid=26524, ip=127.0.0.1, repr=<ray.rllib.evaluation.rollout_worker.RolloutWorker object at 0x000001E85400C850>)
  File "python\ray\_raylet.pyx", line 528, in ray._raylet.raise_if_dependency_failed
ray.exceptions.RaySystemError: System error: No module named 'training'
traceback: Traceback (most recent call last):
  File "C:\Users\crens\Documents\GitHub\ac-validation\.venv310\lib\site-packages\ray\_private\serialization.py", line 369, in dese

## Option B: Short Debug Run (1M steps, 1 seed)

Use this to verify the training loop works before committing to full runs.

In [ ]:
# DEBUG: short run to verify Colab setup
DEBUG_STEPS = 20_000

# Uncomment to run:
# logger_debug = run_training(
#     condition=CONDITION,
#     seed=99,  # use seed 99 to not overwrite real results
#     total_timesteps=DEBUG_STEPS,
#     eval_interval=5_000,
#     results_dir=RESULTS_DIR,
# )
print('Debug run commented out. Recommended first Colab smoke test: 20K steps, eval every 5K.')

## Inline Monitoring: Plot Progress During Training

In [ ]:
def plot_seed_progress(condition: str, seed: int, results_dir: str = RESULTS_DIR):
    """Load and plot metrics for a completed seed."""
    path = f'{results_dir}/{condition}_seed{seed}_metrics.npz'
    if not os.path.exists(path):
        print(f'Not found: {path}')
        return
    
    logger = MetricsLogger.load(path)
    arrays = logger.to_arrays()
    steps = arrays.get('step', np.array([]))
    
    metrics_to_plot = ['floor_utility_mean', 'total_utility_mean', 'gini_wealth_mean']
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    
    for ax, metric in zip(axes, metrics_to_plot):
        if metric in arrays:
            ax.plot(steps, arrays[metric], color='#e74c3c', linewidth=2)
            ax.set_xlabel('Training Steps')
            ax.set_ylabel(metric.replace('_', ' '))
            ax.set_title(metric)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
    
    plt.suptitle(f'SUM Condition — Seed {seed}', y=1.02)
    plt.tight_layout()
    plt.show()

# Plot any completed seeds
for s in range(N_SEEDS):
    plot_seed_progress(CONDITION, s)

## Notes on Expected Results

Under SUM, the planner maximizes total coin. Since skills are Pareto-distributed,
high-skill agents earn much more. The optimal SUM policy extracts from low-skill
(floor) agents and redistributes upward, or simply ignores them.

Expected trajectory:
- Steps 0-2M: All agents improve (coordination phase)
- Steps 2M-10M: Gini rises, floor utility plateaus or declines
- Final: max total utility but high Gini and low floor utility

If floor utility rises monotonically, check that the planner reward is correctly
set to SUM and not accidentally using JAM or NASH.

**Next:** Notebook 05 — NASH condition.